# Fire-product comparison — run & inspect

This notebook drives `peatfire.fire_products_comparison` and **inspects the data at every step**
so you can see exactly what each transformation produces.

## The data-transformation pipeline (one worked example: MCD64A1, 2017)

```
RAW                     PREPROCESS (download_and_clip_data.ipynb)        THIS NOTEBOOK / TOOLKIT
---                     -----------------------------------------        -----------------------
MCD64A1 HDF4 tiles      mosaic h11v05+h12v05 -> clip to NC ->            load_standardized("MCD64A1", 2017, aoi)
(h11v05, h12v05),       save monthly GeoTIFFs:                            -> clip each monthly tif to the AOI (in memory)
sinusoidal, 500 m         data/processed/fire/MCD64A1_061/                -> apply burn_predicate (BurnDate > 0) -> bool
                          MCD64A1_A2017001_nc.tif (Jan)                   -> OR the 12 monthly masks -> ANNUAL bool mask
                          MCD64A1_A2017032_nc.tif (Feb) ...                  (still sinusoidal, 500 m, True=burned)
                                                                          |
                                                                          v
                                                          build_common_grid(aoi, 500 m, EPSG:5070)  ->  empty reference grid
                                                                          |
                                                          to_common_grid(mask, grid, how="max")  ->  mask on the SHARED grid
                                                                          |
                                          +-------------------------------+-------------------------------+
                                          v                               v                               v
                              burned_area_km2(mask)         agreement_matrix(...) pools          plot_overlay_map(stack, pair)
                              -> km^2 for the year          cells across products -> Jaccard/kappa
```

Key invariants:
- **Analysis CRS is always EPSG:5070** (equal-area) for area and grid comparison; the AOI's CRS is only a clip mask.
- The **common grid** (default 500 m, `how="max"` = "any sub-cell burn lights the cell") removes resolution as a confound.
- **Annual mode** ORs a product's monthly files into one yearly mask; **monthly mode** keeps each month separate and drops annual-only products.

Run the cells top to bottom. Each prints/plots the intermediate object it just made.

## 0. Setup & area of interest (AOI)

In [ ]:
%load_ext autoreload
%autoreload 2

import geopandas as gpd
import matplotlib.pyplot as plt

from peatfire import data_path
from peatfire.preproc import clip_raster_to_mask, clip_vector_to_mask
from peatfire.fire_products_comparison import (
    FIRE_PRODUCTS, list_products, get_spec, load_standardized,
    build_common_grid, to_common_grid, burned_area_km2,
    annual_burned_area_series, stack_on_common_grid, agreement_matrix,
    compare_fire_products,
    set_fire_style, plot_annual_series, plot_agreement_heatmap, plot_overlay_map,
)

set_fire_style()

# AOI: swap this path for NC peatlands / non-peatlands later -- nothing else changes.
aoi = gpd.read_file(data_path("processed", "boundaries", "nc_boundary.gpkg"))
print("AOI CRS      :", aoi.crs)
print("AOI bounds   :", aoi.total_bounds)
print("AOI area km^2:", round(float(aoi.to_crs('EPSG:5070').area.sum()) / 1e6, 1))
aoi.plot(facecolor="none", edgecolor="black"); plt.title("AOI"); plt.show()

## 1. What products are registered, and which have data on disk?

The registry is the single source of truth. `directory` resolves each product's folder via `data_path`.

In [ ]:
import pandas as pd
rows = []
for name in list_products():
    s = get_spec(name)
    n_files = len(sorted(s.directory.glob(s.glob))) if s.directory.exists() else 0
    rows.append({"product": name, "family": s.family, "res_m": s.native_res_m,
                 "temporal": s.temporal, "monthly": s.month_parser is not None,
                 "dir_exists": s.directory.exists(), "n_files": n_files,
                 "directory": str(s.directory)})
pd.DataFrame(rows).set_index("product")

## 2. Load ONE product for ONE year, and inspect every sub-step

We trace MCD64A1 / 2017. First the raw clip (note it keeps a singleton `band` dim), then the
standardized annual mask the toolkit actually uses (band squeezed, months OR'd).

In [ ]:
product, year = "MCD64A1", 2017
spec = get_spec(product)

# (a) the monthly files the loader will group for this year
files = sorted(spec.directory.glob(spec.glob))
files_2017 = [f for f in files if spec.year_parser(f) == year]
print(f"{product}: {len(files_2017)} monthly files for {year}")
for f in files_2017[:3]:
    print("   ", f.name, "-> month", spec.month_parser(f))

# (b) raw clip of a single month (in memory). Still has a band dimension.
one = clip_raster_to_mask(files_2017[0], aoi)
print("\nsingle clipped month -> dims:", dict(one.sizes), "| CRS:", one.rio.crs)

In [ ]:
# (c) the standardized ANNUAL mask (band squeezed, 12 months OR'd, True=burned)
mask = load_standardized(product, year, aoi)
print("annual mask name :", mask.name)
print("dims             :", dict(mask.sizes))
print("CRS              :", mask.rio.crs, "(native sinusoidal)")
print("burned pixels    :", int((mask.values == True).sum()))
mask.astype("uint8").plot(); plt.title(f"{product} {year} annual burned mask (native grid)"); plt.show()

## 3. The common grid (the resolution-confound fix)

`build_common_grid` makes an empty EPSG:5070 reference grid over the AOI; `to_common_grid` warps any
product onto it with `how="max"`. Every product is matched to THIS grid, so none is privileged.

In [ ]:
grid = build_common_grid(aoi, res_m=500.0)
print("common grid CRS  :", grid.rio.crs)
print("common grid shape:", dict(grid.sizes))
print("pixel size (m)   :", grid.rio.resolution())

mask_cg = to_common_grid(mask.astype("float32"), grid, how="max") > 0
print("\nmask on common grid -> dims:", dict(mask_cg.sizes), "| CRS:", mask_cg.rio.crs)
mask_cg.astype("uint8").plot(); plt.title(f"{product} {year} on 500 m EPSG:5070 common grid"); plt.show()

## 4. Annual burned-area series — native vs common-grid (Humber Figure 3)

`mode="native"` uses each product's own pixel area; `mode="common_grid"` puts everyone on the shared
500 m grid first. Comparing the two shows how much of any product difference is just resolution.

In [ ]:
products_ba = ["MCD64A1", "GABAM", "FireCCI51", "USGS_BA"]   # absent ones skip with a warning
years = range(2001, 2022)

area_native = annual_burned_area_series(products_ba, years, aoi, mode="native")
area_cg     = annual_burned_area_series(products_ba, years, aoi, mode="common_grid")
display(area_native.head())
display(area_cg.head())

In [ ]:
plot_annual_series(area_native, title="NC burned area — native resolution"); plt.show()
plot_annual_series(area_cg,     title="NC burned area — 500 m common grid"); plt.show()

## 5. Agreement matrices ("confusion-matrix" heatmaps)

VIIRS is included as the independent occurrence check. Binary spatial agreement (Jaccard, kappa) and
temporal correlation of annual totals answer different questions.

In [ ]:
products_all = ["MCD64A1", "GABAM", "FireCCI51", "USGS_BA", "VIIRS"]

jac = agreement_matrix(products_all, years, aoi, method="jaccard", pooling="cells")
kap = agreement_matrix(products_all, years, aoi, method="kappa",   pooling="cells")
cor = agreement_matrix(products_ba,  years, aoi, method="pearson", pooling="years")
display(jac.round(2)); display(cor.round(2))

In [ ]:
plot_agreement_heatmap(jac, title="Spatial agreement (Jaccard)", vmin=0, vmax=1); plt.show()
plot_agreement_heatmap(cor, title="Annual-total correlation (Pearson)", vmin=-1, vmax=1); plt.show()

## 6. Overlay map for a chosen year

`stack_on_common_grid` aligns every product for the year; `plot_overlay_map` shows where a pair agree
(1 = A only, 2 = B only, 3 = both).

In [ ]:
stack_2017 = stack_on_common_grid(products_ba, 2017, aoi, binary=True)
print("products present in 2017 stack:", list(stack_2017))
if {"GABAM", "MCD64A1"} <= set(stack_2017):
    plot_overlay_map(stack_2017, aoi, 2017, pair=("GABAM", "MCD64A1")); plt.show()

## 7. Monthly mode

One flag: `temporal_unit="month"`. Annual-only products (GABAM, USGS_BA, ...) are dropped; the monthly
products + VIIRS remain. The series index becomes month-start Timestamps.

In [ ]:
area_monthly = annual_burned_area_series(
    ["MCD64A1", "FireCCI51", "FireCCIS311"], range(2017, 2019),
    aoi, mode="common_grid", temporal_unit="month",
)
display(area_monthly.head(14))
plot_annual_series(area_monthly, title="Monthly burned area (2017-2018)"); plt.show()

## 8. One-call full comparison (writes CSVs)

`compare_fire_products` runs the whole thing and (with `out_dir`) saves every table. Point `aoi` at a
peatland shapefile later and rerun — nothing else changes.

In [ ]:
res = compare_fire_products(
    data_path("processed", "boundaries", "nc_boundary.gpkg"),
    products=["MCD64A1", "GABAM", "FireCCI51", "USGS_BA", "VIIRS"],
    years=range(2001, 2022),
    agreement_methods=("jaccard", "kappa"),
    overlay_years=[2017],
    out_dir=data_path("processed", "fire", "comparison", "nc"),
)
print("result keys:", list(res))
display(res["area_common_grid"].head())
display(res["agreement_jaccard"].round(2))

In [ ]:
# monthly full run (annual-only products auto-dropped)
res_m = compare_fire_products(
    data_path("processed", "boundaries", "nc_boundary.gpkg"),
    years=range(2017, 2019),
    temporal_unit="month",
    overlay_years=[(2017, 6)],
    out_dir=data_path("processed", "fire", "comparison", "nc_monthly"),
)
print("monthly result keys:", list(res_m))
display(res_m["area_common_grid"].head(14))